In [1]:
import os
import time
from glob import glob
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from data import DriveDataset
from model import build_unet, train, evaluate
from loss import DiceLoss, DiceBCELoss
from utils import seeding, create_dir, epoch_time
import random




In [2]:
seeding(42)

## Loading the Dataset

In [3]:
# Create a directory named "files" if it doesn't already exist
create_dir("files")

In [4]:
mask_folder = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/Mask_tiles/train_masks/'
DSM_folder = '/Users/nirgut/Documents/Master_Files/pipeline_testing/crop/DSM_tiles/normalized/train_data'
# Load and sort dataset paths
all_train_y = sorted(glob(os.path.join(mask_folder, "*.tif")))
all_train_x = sorted(glob(os.path.join(DSM_folder, "*.tif")))

# Print dataset size
data_str = f"Dataset Size:\nTrain: {len(all_train_x)}\n"
print(data_str)


Dataset Size:
Train: 38



## Setting Hyperparameters

In [5]:
# Define model and training parameters
H, W = 256, 256
size = (H, W)
batch_size = 64
num_epochs = 30
lr = 1e-4
resume_from_checkpoint = False
checkpoint_folder_path = "files/"
checkpoint_path = "/Users/nirgut/ML_Course/2nd_unet/files/two_features_epoch_29.pth"

## Splitting Data into Training and Validation Sets

In [6]:
# Split data into training and validation sets based on the defined ratio
train_ratio = 0.8
num_train_samples = int(train_ratio * len(all_train_x))

train_indices = random.sample(range(len(all_train_x)), num_train_samples)
valid_indices = list(set(range(len(all_train_x))) - set(train_indices))

train_x = [all_train_x[i] for i in train_indices]
# train_x2 = [all_train_x2[i] for i in train_indices]
train_y = [all_train_y[i] for i in train_indices]

valid_x = [all_train_x[i] for i in valid_indices]
# valid_x2 = [all_train_x2[i] for i in valid_indices]
valid_y = [all_train_y[i] for i in valid_indices]

print(f'Train: {len(train_x)} - {len(train_y)}')
print(f'Valid: {len(valid_x)} - {len(valid_y)}')



Train: 30 - 30
Valid: 8 - 8


## Creating Dataset and DataLoader

In [7]:
# Create custom datasets and dataloaders for training and validation
train_dataset = DriveDataset(train_x, train_y)
valid_dataset = DriveDataset(valid_x, valid_y)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    dataset=valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2
)

In [8]:
# Section 8: Setting Up the Device and Model
# Check for CUDA availability and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize and transfer the model to the device
model = build_unet().to(device)

In [9]:
# Section 9: Defining Optimizer, Scheduler, and Loss Function
# Set up the optimizer, learning rate scheduler, and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, verbose=True)
loss_fn = DiceBCELoss()

In [10]:
# Section 10: Loading Checkpoint
# Load model checkpoint if resume_from_checkpoint is True and checkpoint exists
if resume_from_checkpoint and os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path))
    print("Checkpoint loaded successfully. Resuming training.")



In [11]:
# Section 11: Training the Model
# Train and validate the model for the specified number of epochs
best_valid_loss = float("inf")

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss = train(model, train_loader, optimizer, loss_fn, device)
    valid_loss = evaluate(model, valid_loader, loss_fn, device)
    epoch_checkpoint_path = f"{checkpoint_folder_path}/two_features_epoch_{epoch+1}.pth"

    # Saving the model if validation loss improves
    if valid_loss < best_valid_loss:
        print(f"Valid loss improved from {best_valid_loss:2.4f} to {valid_loss:2.4f}. Saving checkpoint: {epoch_checkpoint_path}")
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), epoch_checkpoint_path)

    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    # Print epoch results
    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f}')
    print(f'\t Val. Loss: {valid_loss:.3f}')

Valid loss improved from inf to 1.6835. Saving checkpoint: files//two_features_epoch_1.pth
Epoch: 01 | Epoch Time: 1m 53s
	Train Loss: 1.749
	 Val. Loss: 1.684
Epoch: 02 | Epoch Time: 1m 50s
	Train Loss: 1.699
	 Val. Loss: 1.690
Epoch: 03 | Epoch Time: 1m 46s
	Train Loss: 1.668
	 Val. Loss: 1.695
Epoch: 04 | Epoch Time: 1m 48s
	Train Loss: 1.645
	 Val. Loss: 1.699
Epoch: 05 | Epoch Time: 1m 47s
	Train Loss: 1.624
	 Val. Loss: 1.702
Epoch: 06 | Epoch Time: 1m 46s
	Train Loss: 1.606
	 Val. Loss: 1.705
Epoch: 07 | Epoch Time: 1m 47s
	Train Loss: 1.591
	 Val. Loss: 1.707
Epoch: 08 | Epoch Time: 1m 47s
	Train Loss: 1.578
	 Val. Loss: 1.708
Epoch: 09 | Epoch Time: 1m 46s
	Train Loss: 1.565
	 Val. Loss: 1.709


KeyboardInterrupt: 